<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/SLM-Finetuning-Demo/blob/main/SLM_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip uninstall -y peft transformers unsloth
!pip install unsloth peft>=0.12 transformers datasets trl accelerate bitsandbytes xformers

Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
Found existing installation: unsloth 2026.1.2
Uninstalling unsloth-2026.1.2:
  Successfully uninstalled unsloth-2026.1.2


In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
  from unsloth import FastLanguageModel
  from datasets import load_dataset
  from transformers import TrainingArguments
  from trl import SFTTrainer

  max_seq_length = 2048

  model, tokenizer = FastLanguageModel.from_pretrained(
      model_name = "unsloth/mistral-7b-bnb-4bit",
      max_seq_length = max_seq_length,
      load_in_4bit = True
  )

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
)

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


In [4]:
dataset = load_dataset("json", data_files="arise_finetune_dataset_2196.json")

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
def format_prompt(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}{tokenizer.eos_token}"""
    }

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/2196 [00:00<?, ? examples/s]

In [6]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 10,
        output_dir = "nbfc_model",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2196 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,196 | Num Epochs = 3 | Total steps = 825
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,631,488 of 7,255,363,584 (0.19% trained)


Step,Training Loss
10,1.758410
20,0.763140
30,0.542188
40,0.391585
50,0.315675
60,0.273341
70,0.238808
80,0.227098
90,0.205923
100,0.172288


TrainOutput(global_step=825, training_loss=0.18410326112400402, metrics={'train_runtime': 1796.5589, 'train_samples_per_second': 3.667, 'train_steps_per_second': 0.459, 'total_flos': 1.4767716047142912e+16, 'train_loss': 0.18410326112400402, 'epoch': 3.0})

In [7]:
model.save_pretrained("nbfc_model")
tokenizer.save_pretrained("nbfc_model")
!zip -r nbfc_model.zip nbfc_model

  adding: nbfc_model/ (stored 0%)
  adding: nbfc_model/checkpoint-825/ (stored 0%)
  adding: nbfc_model/checkpoint-825/trainer_state.json (deflated 77%)
  adding: nbfc_model/checkpoint-825/scheduler.pt (deflated 62%)
  adding: nbfc_model/checkpoint-825/scaler.pt (deflated 64%)
  adding: nbfc_model/checkpoint-825/tokenizer_config.json (deflated 48%)
  adding: nbfc_model/checkpoint-825/training_args.bin (deflated 53%)
  adding: nbfc_model/checkpoint-825/README.md (deflated 65%)
  adding: nbfc_model/checkpoint-825/tokenizer.json (deflated 85%)
  adding: nbfc_model/checkpoint-825/optimizer.pt (deflated 8%)
  adding: nbfc_model/checkpoint-825/rng_state.pth (deflated 27%)
  adding: nbfc_model/checkpoint-825/adapter_config.json (deflated 58%)
  adding: nbfc_model/checkpoint-825/adapter_model.safetensors (deflated 8%)
  adding: nbfc_model/tokenizer_config.json (deflated 48%)
  adding: nbfc_model/README.md (deflated 44%)
  adding: nbfc_model/tokenizer.json (deflated 85%)
  adding: nbfc_model/ch

In [11]:
!pip uninstall transformers peft -y
!pip install transformers==4.40.2 peft==0.10.0 accelerate bitsandbytes
!pip install unsloth

Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 6.0 MB/s eta 0:00:00
  Using cached peft-0.10.0-py3-none-any.whl.metadata (13 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 83.9 MB/s eta 0:00:00
Using cached peft-0.10.0-py3-none-any.whl (199 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstal

  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.6.0-py3-none-any.whl.metadata (13 kB)
Using cached peft-0.18.1-py3-none-any.whl (556 kB)
Using cached transformers-5.2.0-py3-none-any.whl (10.4 MB)
Using cached huggingface_hub-1.6.0-py3-none-any.whl (612 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.40.2
    Uninstalling transformers-4.40.2:
      Successfully uninstalled transformers-4.40.2
  Attemptin

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "unsloth/mistral-7b-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto"
)

model = PeftModel.from_pretrained(model, "nbfc_model")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [2]:
prompt = """
### Instruction:
அரைஸ் HO Enga iruku ?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.1
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



### Instruction:
அரைஸ் HO Enga iruku ?

### Response:
Arise Capital provides loans mainly to women in rural and semi-rural areas.
